# 06. Entrenamiento V3 — XGBoost vs CatBoost

## Objetivo

Entrenar y comparar modelos sobre el dataset V2 generado en el notebook 04, utilizando **todas las variables válidas** y evitando la selección agresiva previa de variables.

Esta versión corrige el problema de compatibilidad entre `CatBoostClassifier` y `sklearn.clone()`:

- **CatBoost nunca se pasa a `cross_validate`, `cross_val_predict` ni `RandomizedSearchCV`.**
- `cat_features` **no se incluye en el constructor** de `CatBoostClassifier`.
- `cat_features` se pasa únicamente en `model.fit(...)`.
- La validación cruzada, el tuning y las predicciones OOF de CatBoost se realizan manualmente.

### Modelos comparados

1. XGBoost.
2. CatBoost con variables categóricas nativas.
3. Balanced Random Forest como referencia.

### Métrica principal

**PR-AUC / Average Precision**, debido al fuerte desbalance de clases.

El holdout V3 se mantiene fuera de todas las decisiones de tuning y threshold hasta la evaluación final.


In [ ]:
from pathlib import Path
import json
import warnings

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from IPython.display import display
from scipy.stats import randint, uniform, loguniform

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score,
    average_precision_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    fbeta_score,
    make_scorer,
    matthews_corrcoef,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve
)
from sklearn.model_selection import (
    ParameterSampler,
    RandomizedSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
    cross_val_predict,
    cross_validate,
    train_test_split
)
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

from imblearn.ensemble import BalancedRandomForestClassifier
from xgboost import XGBClassifier
from catboost import CatBoostClassifier


warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 250)
pd.set_option("display.max_rows", 200)
pd.set_option("display.float_format", lambda value: f"{value:,.4f}")


NOTEBOOK_VERSION = "06_MODEL_TRAINING_V3_CATBOOST_MANUAL"

RANDOM_STATE = 42
FINAL_SPLIT_RANDOM_STATE = 20260825

TARGET = "incident_liver_disease_10y"
ID_COLUMN = "pid"
WEIGHT_COLUMN = "wgt_c"

TEST_SIZE = 0.20

N_SPLITS = 5
N_REPEATS = 3
TUNING_SPLITS = 5

SEARCH_ITERATIONS_XGB = 25
SEARCH_ITERATIONS_CAT = 20

MIN_CATEGORY_FREQUENCY = 10
MINIMUM_RECALL = 0.70

print(NOTEBOOK_VERSION)


In [ ]:
def find_project_root(start_path: Path) -> Path:
    start_path = start_path.resolve()

    for candidate in [start_path, *start_path.parents]:
        if (
            (candidate / "data").exists()
            and (candidate / "notebooks").exists()
        ):
            return candidate

    raise FileNotFoundError(
        "No se ha encontrado la raíz del proyecto."
    )


PROJECT_ROOT = find_project_root(Path.cwd())

DATASET_PATH = (
    PROJECT_ROOT
    / "data"
    / "processed"
    / "klosa_liver_modeling_dataset_v2.csv"
)

VARIABLE_GROUPS_PATH = (
    PROJECT_ROOT
    / "configs"
    / "variable_groups_v2.json"
)

REPORTS_DIR = (
    PROJECT_ROOT
    / "reports"
    / "model_training_v3"
)

MODELS_DIR = PROJECT_ROOT / "models"

REPORTS_DIR.mkdir(parents=True, exist_ok=True)
MODELS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Proyecto: {PROJECT_ROOT}")
print(f"Dataset: {DATASET_PATH}")
print(f"Variables: {VARIABLE_GROUPS_PATH}")


In [ ]:
for path in [DATASET_PATH, VARIABLE_GROUPS_PATH]:
    if not path.exists():
        raise FileNotFoundError(
            f"No se ha encontrado:\n{path}"
        )

df = pd.read_csv(
    DATASET_PATH,
    low_memory=False
)

with open(
    VARIABLE_GROUPS_PATH,
    "r",
    encoding="utf-8"
) as file:
    variable_groups = json.load(file)

continuous_columns = [
    column
    for column in variable_groups["continuous_columns"]
    if (
        column in df.columns
        and column not in {TARGET, ID_COLUMN, WEIGHT_COLUMN}
    )
]

categorical_columns = [
    column
    for column in variable_groups["categorical_columns"]
    if (
        column in df.columns
        and column not in {TARGET, ID_COLUMN, WEIGHT_COLUMN}
        and column not in continuous_columns
    )
]

feature_columns = list(
    dict.fromkeys(
        continuous_columns + categorical_columns
    )
)

if ID_COLUMN not in df.columns or TARGET not in df.columns:
    raise KeyError(
        f"El dataset debe contener {ID_COLUMN} y {TARGET}."
    )

invalid_target_values = (
    set(df[TARGET].dropna().unique()) - {0, 1}
)

if invalid_target_values:
    raise ValueError(
        f"Target inválido: {invalid_target_values}"
    )

print("DATASET V2")
print("=" * 70)
print(f"Dimensiones: {df.shape}")
print(f"Predictores: {len(feature_columns)}")
print(f"Continuas: {len(continuous_columns)}")
print(f"Categóricas: {len(categorical_columns)}")
print(f"PID duplicados: {df[ID_COLUMN].duplicated().sum():,}")
print(f"Positivos: {int(df[TARGET].sum()):,}")
print(f"Prevalencia: {df[TARGET].mean():.2%}")

assert df[ID_COLUMN].duplicated().sum() == 0
assert df[TARGET].isna().sum() == 0


## 1. División development / holdout V3

Todas las decisiones de selección del modelo, tuning y threshold se toman únicamente sobre `development`.


In [ ]:
X = df[feature_columns].copy()
y = df[TARGET].astype("int8").copy()

development_indices, holdout_indices = train_test_split(
    df.index.to_numpy(),
    test_size=TEST_SIZE,
    stratify=y,
    random_state=FINAL_SPLIT_RANDOM_STATE
)

X_dev = X.loc[development_indices].copy()
X_holdout = X.loc[holdout_indices].copy()

y_dev = y.loc[development_indices].copy()
y_holdout = y.loc[holdout_indices].copy()

split_report = df[[ID_COLUMN, TARGET]].copy()
split_report["split_v3"] = "holdout_v3"
split_report.loc[
    development_indices,
    "split_v3"
] = "development"

split_report.to_csv(
    REPORTS_DIR / "development_holdout_split_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

print("DIVISIÓN V3")
print("=" * 70)
print(f"Development: {X_dev.shape}")
print(f"Holdout V3: {X_holdout.shape}")
print(f"Positivos development: {int(y_dev.sum())}")
print(f"Positivos holdout: {int(y_holdout.sum())}")
print(f"Prevalencia development: {y_dev.mean():.2%}")
print(f"Prevalencia holdout: {y_holdout.mean():.2%}")


In [ ]:
class_counts = y_dev.value_counts().sort_index()

n_negative = int(class_counts.get(0, 0))
n_positive = int(class_counts.get(1, 0))

imbalance_ratio = n_negative / n_positive

print("DESBALANCE DEVELOPMENT")
print("=" * 70)
print(f"Negativos: {n_negative:,}")
print(f"Positivos: {n_positive:,}")
print(f"Ratio negativos/positivos: {imbalance_ratio:.2f}:1")


## 2. Preprocesamiento para XGBoost y Balanced Random Forest


In [ ]:
def make_onehot_encoder():
    try:
        return OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=MIN_CATEGORY_FREQUENCY,
            sparse_output=True
        )
    except TypeError:
        return OneHotEncoder(
            handle_unknown="ignore",
            min_frequency=MIN_CATEGORY_FREQUENCY,
            sparse=True
        )


def build_matrix_preprocessor():
    transformers = []

    if continuous_columns:
        continuous_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="median",
                        add_indicator=True
                    )
                ),
                (
                    "scaler",
                    StandardScaler()
                )
            ]
        )

        transformers.append(
            (
                "continuous",
                continuous_pipeline,
                continuous_columns
            )
        )

    if categorical_columns:
        categorical_pipeline = Pipeline(
            steps=[
                (
                    "imputer",
                    SimpleImputer(
                        strategy="most_frequent"
                    )
                ),
                (
                    "onehot",
                    make_onehot_encoder()
                )
            ]
        )

        transformers.append(
            (
                "categorical",
                categorical_pipeline,
                categorical_columns
            )
        )

    return ColumnTransformer(
        transformers=transformers,
        remainder="drop"
    )


matrix_preprocessor = build_matrix_preprocessor()

print("Preprocessor XGBoost/RF creado.")


## 3. Preparación CatBoost

`cat_features` se pasará **solo** a `fit()`.


In [ ]:
def prepare_catboost_frame(X_frame: pd.DataFrame) -> pd.DataFrame:
    X_cat = X_frame.copy()

    for column in continuous_columns:
        X_cat[column] = pd.to_numeric(
            X_cat[column],
            errors="coerce"
        )

    for column in categorical_columns:
        X_cat[column] = (
            X_cat[column]
            .astype("string")
            .fillna("__MISSING__")
            .astype(str)
        )

    return X_cat


X_dev_catboost = prepare_catboost_frame(
    X_dev
)

X_holdout_catboost = prepare_catboost_frame(
    X_holdout
)

print(
    f"CatBoost development: {X_dev_catboost.shape}"
)


In [ ]:
repeated_cv = RepeatedStratifiedKFold(
    n_splits=N_SPLITS,
    n_repeats=N_REPEATS,
    random_state=RANDOM_STATE
)

tuning_cv = StratifiedKFold(
    n_splits=TUNING_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE
)

scoring = {
    "pr_auc": "average_precision",
    "roc_auc": "roc_auc",
    "precision": make_scorer(
        precision_score,
        zero_division=0
    ),
    "recall": make_scorer(
        recall_score,
        zero_division=0
    ),
    "f1": make_scorer(
        f1_score,
        zero_division=0
    ),
    "balanced_accuracy": make_scorer(
        balanced_accuracy_score
    ),
    "mcc": make_scorer(
        matthews_corrcoef
    )
}

print(
    f"Benchmark: {N_SPLITS} folds × {N_REPEATS} repeticiones"
)


In [ ]:
def evaluate_catboost_cv(
    X_data,
    y_data,
    params,
    cv
):
    fold_records = []

    for fold_number, (train_idx, valid_idx) in enumerate(
        cv.split(X_data, y_data),
        start=1
    ):
        X_fold_train = X_data.iloc[train_idx]
        X_fold_valid = X_data.iloc[valid_idx]

        y_fold_train = y_data.iloc[train_idx]
        y_fold_valid = y_data.iloc[valid_idx]

        model = CatBoostClassifier(
            **params
        )

        model.fit(
            X_fold_train,
            y_fold_train,
            cat_features=categorical_columns
        )

        probabilities = model.predict_proba(
            X_fold_valid
        )[:, 1]

        predictions = (
            probabilities >= 0.5
        ).astype(int)

        fold_records.append({
            "fold": fold_number,
            "pr_auc": average_precision_score(
                y_fold_valid,
                probabilities
            ),
            "roc_auc": roc_auc_score(
                y_fold_valid,
                probabilities
            ),
            "precision": precision_score(
                y_fold_valid,
                predictions,
                zero_division=0
            ),
            "recall": recall_score(
                y_fold_valid,
                predictions,
                zero_division=0
            ),
            "f1": f1_score(
                y_fold_valid,
                predictions,
                zero_division=0
            ),
            "balanced_accuracy": balanced_accuracy_score(
                y_fold_valid,
                predictions
            ),
            "mcc": matthews_corrcoef(
                y_fold_valid,
                predictions
            )
        })

    fold_df = pd.DataFrame(
        fold_records
    )

    summary = {
        "pr_auc_mean": fold_df["pr_auc"].mean(),
        "pr_auc_std": fold_df["pr_auc"].std(),
        "roc_auc_mean": fold_df["roc_auc"].mean(),
        "roc_auc_std": fold_df["roc_auc"].std(),
        "precision_mean": fold_df["precision"].mean(),
        "recall_mean": fold_df["recall"].mean(),
        "f1_mean": fold_df["f1"].mean(),
        "balanced_accuracy_mean": (
            fold_df["balanced_accuracy"].mean()
        ),
        "mcc_mean": fold_df["mcc"].mean()
    }

    return summary, fold_df


def catboost_oof_probabilities(
    X_data,
    y_data,
    params,
    cv
):
    probabilities = np.full(
        len(X_data),
        np.nan,
        dtype=float
    )

    for train_idx, valid_idx in cv.split(
        X_data,
        y_data
    ):
        model = CatBoostClassifier(
            **params
        )

        model.fit(
            X_data.iloc[train_idx],
            y_data.iloc[train_idx],
            cat_features=categorical_columns
        )

        probabilities[valid_idx] = (
            model.predict_proba(
                X_data.iloc[valid_idx]
            )[:, 1]
        )

    if np.isnan(probabilities).any():
        raise RuntimeError(
            "Faltan predicciones OOF de CatBoost."
        )

    return probabilities


## 4. Benchmark inicial


In [ ]:
xgb_baseline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(matrix_preprocessor)
        ),
        (
            "model",
            XGBClassifier(
                n_estimators=600,
                learning_rate=0.04,
                max_depth=3,
                min_child_weight=3,
                subsample=0.80,
                colsample_bytree=0.80,
                reg_lambda=2.0,
                scale_pos_weight=imbalance_ratio,
                objective="binary:logistic",
                eval_metric="aucpr",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=1
            )
        )
    ]
)

balanced_rf_baseline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(matrix_preprocessor)
        ),
        (
            "model",
            BalancedRandomForestClassifier(
                n_estimators=600,
                max_depth=8,
                min_samples_leaf=2,
                random_state=RANDOM_STATE,
                n_jobs=1,
                replacement=True
            )
        )
    ]
)

catboost_baseline_params = {
    "iterations": 600,
    "depth": 5,
    "learning_rate": 0.04,
    "loss_function": "Logloss",
    "auto_class_weights": "Balanced",
    "random_seed": RANDOM_STATE,
    "verbose": 0,
    "allow_writing_files": False,
    "thread_count": 1
}

print("Baselines preparados.")


In [ ]:
benchmark_records = []
benchmark_fold_records = []

# XGBoost y Balanced RF
for model_name, estimator in {
    "XGBoost_AllFeatures": xgb_baseline,
    "BalancedRandomForest": balanced_rf_baseline
}.items():

    print(f"Evaluando {model_name}...")

    results = cross_validate(
        estimator=estimator,
        X=X_dev,
        y=y_dev,
        cv=repeated_cv,
        scoring=scoring,
        n_jobs=-1,
        return_train_score=False,
        error_score="raise"
    )

    benchmark_records.append({
        "modelo": model_name,
        "pr_auc_mean": results["test_pr_auc"].mean(),
        "pr_auc_std": results["test_pr_auc"].std(),
        "roc_auc_mean": results["test_roc_auc"].mean(),
        "roc_auc_std": results["test_roc_auc"].std(),
        "precision_mean": results["test_precision"].mean(),
        "recall_mean": results["test_recall"].mean(),
        "f1_mean": results["test_f1"].mean(),
        "balanced_accuracy_mean": (
            results["test_balanced_accuracy"].mean()
        ),
        "mcc_mean": results["test_mcc"].mean()
    })

    for fold_index in range(
        len(results["test_pr_auc"])
    ):
        benchmark_fold_records.append({
            "modelo": model_name,
            "fold": fold_index + 1,
            "pr_auc": results["test_pr_auc"][fold_index],
            "roc_auc": results["test_roc_auc"][fold_index],
            "recall": results["test_recall"][fold_index],
            "precision": results["test_precision"][fold_index],
            "f1": results["test_f1"][fold_index],
            "mcc": results["test_mcc"][fold_index]
        })


# CatBoost — MANUAL
print("Evaluando CatBoost_NativeCategorical...")

cat_summary, cat_folds = evaluate_catboost_cv(
    X_dev_catboost,
    y_dev,
    catboost_baseline_params,
    repeated_cv
)

benchmark_records.append({
    "modelo": "CatBoost_NativeCategorical",
    **cat_summary
})

cat_folds = cat_folds.copy()
cat_folds["modelo"] = (
    "CatBoost_NativeCategorical"
)

benchmark_fold_records.extend(
    cat_folds[
        [
            "modelo",
            "fold",
            "pr_auc",
            "roc_auc",
            "recall",
            "precision",
            "f1",
            "mcc"
        ]
    ].to_dict("records")
)

benchmark_results = (
    pd.DataFrame(benchmark_records)
    .sort_values(
        "pr_auc_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

benchmark_folds = pd.DataFrame(
    benchmark_fold_records
)

display(benchmark_results)


In [ ]:
plot_data = benchmark_results.sort_values(
    "pr_auc_mean"
)

plt.figure(figsize=(9, 5))

plt.barh(
    plot_data["modelo"],
    plot_data["pr_auc_mean"],
    xerr=plot_data["pr_auc_std"]
)

plt.xlabel("PR-AUC media")
plt.ylabel("Modelo")
plt.title("Benchmark V3")
plt.tight_layout()

plt.savefig(
    REPORTS_DIR / "benchmark_pr_auc_v3.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()


## 5. Optimización XGBoost


In [ ]:
xgb_search_pipeline = Pipeline(
    steps=[
        (
            "preprocessor",
            clone(matrix_preprocessor)
        ),
        (
            "model",
            XGBClassifier(
                objective="binary:logistic",
                eval_metric="aucpr",
                tree_method="hist",
                random_state=RANDOM_STATE,
                n_jobs=1
            )
        )
    ]
)

xgb_parameter_space = {
    "model__n_estimators": randint(250, 1400),
    "model__learning_rate": loguniform(0.008, 0.20),
    "model__max_depth": randint(2, 7),
    "model__min_child_weight": randint(1, 20),
    "model__subsample": uniform(0.60, 0.40),
    "model__colsample_bytree": uniform(0.50, 0.50),
    "model__gamma": uniform(0, 5),
    "model__reg_alpha": loguniform(1e-4, 15),
    "model__reg_lambda": loguniform(1e-2, 50),
    "model__scale_pos_weight": [
        1.0,
        imbalance_ratio * 0.25,
        imbalance_ratio * 0.50,
        imbalance_ratio * 0.75,
        imbalance_ratio,
        imbalance_ratio * 1.25
    ],
    "model__max_delta_step": [0, 1, 3, 5]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_search_pipeline,
    param_distributions=xgb_parameter_space,
    n_iter=SEARCH_ITERATIONS_XGB,
    scoring="average_precision",
    cv=tuning_cv,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=1,
    refit=True,
    error_score="raise"
)

xgb_search.fit(
    X_dev,
    y_dev
)

print("XGBOOST OPTIMIZADO")
print("=" * 70)
print(f"PR-AUC tuning: {xgb_search.best_score_:.4f}")
display(xgb_search.best_params_)


## 6. Optimización CatBoost — manual


In [ ]:
catboost_parameter_space = {
    "iterations": randint(300, 1400),
    "depth": randint(3, 8),
    "learning_rate": loguniform(0.008, 0.20),
    "l2_leaf_reg": loguniform(0.5, 40),
    "random_strength": loguniform(1e-3, 5),
    "border_count": [64, 128, 254],
    "auto_class_weights": [
        None,
        "Balanced",
        "SqrtBalanced"
    ]
}

parameter_samples = list(
    ParameterSampler(
        catboost_parameter_space,
        n_iter=SEARCH_ITERATIONS_CAT,
        random_state=RANDOM_STATE
    )
)

catboost_search_records = []
best_catboost_score = -np.inf
best_catboost_params = None

for iteration, sampled_params in enumerate(
    parameter_samples,
    start=1
):

    print(
        f"CatBoost tuning "
        f"{iteration}/{len(parameter_samples)}"
    )

    complete_params = {
        **sampled_params,
        "loss_function": "Logloss",
        "random_seed": RANDOM_STATE,
        "verbose": 0,
        "allow_writing_files": False,
        "thread_count": 1
    }

    summary, _ = evaluate_catboost_cv(
        X_dev_catboost,
        y_dev,
        complete_params,
        tuning_cv
    )

    score = float(
        summary["pr_auc_mean"]
    )

    catboost_search_records.append({
        "iteration": iteration,
        "pr_auc": score,
        **sampled_params
    })

    if score > best_catboost_score:
        best_catboost_score = score
        best_catboost_params = (
            complete_params.copy()
        )

catboost_search_results = (
    pd.DataFrame(
        catboost_search_records
    )
    .sort_values(
        "pr_auc",
        ascending=False
    )
    .reset_index(drop=True)
)

print("CATBOOST OPTIMIZADO")
print("=" * 70)
print(
    f"Mejor PR-AUC tuning: "
    f"{best_catboost_score:.4f}"
)
display(
    catboost_search_results.head(10)
)

print("Mejores parámetros:")
display(best_catboost_params)


## 7. Comparación robusta de optimizados


In [ ]:
optimized_records = []

print("Evaluando XGBoost_Optimized...")

xgb_cv_results = cross_validate(
    estimator=xgb_search.best_estimator_,
    X=X_dev,
    y=y_dev,
    cv=repeated_cv,
    scoring=scoring,
    n_jobs=-1,
    error_score="raise"
)

optimized_records.append({
    "modelo": "XGBoost_Optimized",
    "pr_auc_mean": xgb_cv_results["test_pr_auc"].mean(),
    "pr_auc_std": xgb_cv_results["test_pr_auc"].std(),
    "roc_auc_mean": xgb_cv_results["test_roc_auc"].mean(),
    "roc_auc_std": xgb_cv_results["test_roc_auc"].std(),
    "precision_mean": xgb_cv_results["test_precision"].mean(),
    "recall_mean": xgb_cv_results["test_recall"].mean(),
    "f1_mean": xgb_cv_results["test_f1"].mean(),
    "balanced_accuracy_mean": (
        xgb_cv_results["test_balanced_accuracy"].mean()
    ),
    "mcc_mean": xgb_cv_results["test_mcc"].mean()
})

print("Evaluando CatBoost_Optimized...")

cat_opt_summary, cat_opt_folds = evaluate_catboost_cv(
    X_dev_catboost,
    y_dev,
    best_catboost_params,
    repeated_cv
)

optimized_records.append({
    "modelo": "CatBoost_Optimized",
    **cat_opt_summary
})

optimized_results = (
    pd.DataFrame(optimized_records)
    .sort_values(
        "pr_auc_mean",
        ascending=False
    )
    .reset_index(drop=True)
)

display(optimized_results)


In [ ]:
best_model_name = (
    optimized_results.iloc[0]["modelo"]
)

if best_model_name == "XGBoost_Optimized":
    best_model_family = "XGBoost"
    X_dev_final = X_dev
    X_holdout_final = X_holdout

else:
    best_model_family = "CatBoost"
    X_dev_final = X_dev_catboost
    X_holdout_final = X_holdout_catboost

print("MODELO GANADOR")
print("=" * 70)
print(f"Modelo: {best_model_family}")
print(
    "PR-AUC repeated CV: "
    f"{optimized_results.iloc[0]['pr_auc_mean']:.4f} "
    f"± {optimized_results.iloc[0]['pr_auc_std']:.4f}"
)
print(
    "ROC-AUC repeated CV: "
    f"{optimized_results.iloc[0]['roc_auc_mean']:.4f}"
)


## 8. OOF y threshold


In [ ]:
oof_cv = StratifiedKFold(
    n_splits=TUNING_SPLITS,
    shuffle=True,
    random_state=RANDOM_STATE + 101
)

if best_model_family == "XGBoost":

    oof_probabilities = cross_val_predict(
        estimator=xgb_search.best_estimator_,
        X=X_dev_final,
        y=y_dev,
        cv=oof_cv,
        method="predict_proba",
        n_jobs=-1
    )[:, 1]

else:

    oof_probabilities = catboost_oof_probabilities(
        X_dev_final,
        y_dev,
        best_catboost_params,
        oof_cv
    )

oof_pr_auc = average_precision_score(
    y_dev,
    oof_probabilities
)

oof_roc_auc = roc_auc_score(
    y_dev,
    oof_probabilities
)

print(f"PR-AUC OOF: {oof_pr_auc:.4f}")
print(f"ROC-AUC OOF: {oof_roc_auc:.4f}")


In [ ]:
# ============================================================
# ANÁLISIS COMPLETO DE THRESHOLDS SOBRE DEVELOPMENT OOF
# ============================================================

_, _, thresholds = precision_recall_curve(
    y_dev,
    oof_probabilities
)

threshold_records = []

for threshold in thresholds:

    predictions = (
        oof_probabilities >= threshold
    ).astype(int)

    tn, fp, fn, tp = confusion_matrix(
        y_dev,
        predictions
    ).ravel()

    precision = precision_score(
        y_dev,
        predictions,
        zero_division=0
    )

    recall = recall_score(
        y_dev,
        predictions,
        zero_division=0
    )

    specificity = (
        tn / (tn + fp)
        if (tn + fp) > 0
        else 0
    )

    f1 = f1_score(
        y_dev,
        predictions,
        zero_division=0
    )

    f2 = fbeta_score(
        y_dev,
        predictions,
        beta=2,
        zero_division=0
    )

    balanced_acc = balanced_accuracy_score(
        y_dev,
        predictions
    )

    mcc = matthews_corrcoef(
        y_dev,
        predictions
    )

    threshold_records.append({
        "threshold": float(threshold),
        "precision": precision,
        "recall": recall,
        "specificity": specificity,
        "f1": f1,
        "f2": f2,
        "balanced_accuracy": balanced_acc,
        "mcc": mcc,
        "tp": tp,
        "fp": fp,
        "tn": tn,
        "fn": fn
    })


threshold_report = pd.DataFrame(
    threshold_records
)

print(
    f"Thresholds analizados: "
    f"{len(threshold_report):,}"
)

In [ ]:
# ============================================================
# COMPARAR ESTRATEGIAS DE SELECCIÓN DEL THRESHOLD
# ============================================================

strategy_rows = []


def add_strategy(name, row):

    strategy_rows.append({
        "estrategia": name,
        "threshold": row["threshold"],
        "precision": row["precision"],
        "recall": row["recall"],
        "specificity": row["specificity"],
        "f1": row["f1"],
        "f2": row["f2"],
        "balanced_accuracy": row["balanced_accuracy"],
        "mcc": row["mcc"]
    })


# ------------------------------------------------------------
# 1. Máximo MCC
# ------------------------------------------------------------

row_max_mcc = (
    threshold_report
    .sort_values(
        "mcc",
        ascending=False
    )
    .iloc[0]
)

add_strategy(
    "Max MCC",
    row_max_mcc
)


# ------------------------------------------------------------
# 2. Máximo F1
# ------------------------------------------------------------

row_max_f1 = (
    threshold_report
    .sort_values(
        "f1",
        ascending=False
    )
    .iloc[0]
)

add_strategy(
    "Max F1",
    row_max_f1
)


# ------------------------------------------------------------
# 3. Máximo F2
# ------------------------------------------------------------

row_max_f2 = (
    threshold_report
    .sort_values(
        "f2",
        ascending=False
    )
    .iloc[0]
)

add_strategy(
    "Max F2",
    row_max_f2
)


# ------------------------------------------------------------
# 4. Máxima precision manteniendo recall >= 40 %
# ------------------------------------------------------------

candidates_recall_40 = (
    threshold_report
    .loc[
        threshold_report["recall"] >= 0.40
    ]
    .copy()
)

if not candidates_recall_40.empty:

    row_recall_40 = (
        candidates_recall_40
        .sort_values(
            [
                "precision",
                "mcc"
            ],
            ascending=False
        )
        .iloc[0]
    )

    add_strategy(
        "Max Precision | Recall >= 0.40",
        row_recall_40
    )


# ------------------------------------------------------------
# 5. Máxima precision manteniendo recall >= 50 %
# ------------------------------------------------------------

candidates_recall_50 = (
    threshold_report
    .loc[
        threshold_report["recall"] >= 0.50
    ]
    .copy()
)

if not candidates_recall_50.empty:

    row_recall_50 = (
        candidates_recall_50
        .sort_values(
            [
                "precision",
                "mcc"
            ],
            ascending=False
        )
        .iloc[0]
    )

    add_strategy(
        "Max Precision | Recall >= 0.50",
        row_recall_50
    )


threshold_strategies = pd.DataFrame(
    strategy_rows
)

display(
    threshold_strategies
    .sort_values(
        "mcc",
        ascending=False
    )
)

In [ ]:
# ============================================================
# THRESHOLD RECOMENDADO
# ============================================================

MINIMUM_ACCEPTABLE_RECALL = 0.40

balanced_candidates = (
    threshold_report
    .loc[
        threshold_report["recall"]
        >= MINIMUM_ACCEPTABLE_RECALL
    ]
    .copy()
)

if not balanced_candidates.empty:

    best_threshold_row = (
        balanced_candidates
        .sort_values(
            [
                "mcc",
                "precision",
                "specificity"
            ],
            ascending=False
        )
        .iloc[0]
    )

else:

    best_threshold_row = (
        threshold_report
        .sort_values(
            "mcc",
            ascending=False
        )
        .iloc[0]
    )


best_threshold = float(
    best_threshold_row["threshold"]
)


print("NUEVO THRESHOLD SELECCIONADO")
print("=" * 70)

print(
    f"Threshold: "
    f"{best_threshold:.6f}"
)

print(
    f"Precision OOF: "
    f"{best_threshold_row['precision']:.4f}"
)

print(
    f"Recall OOF: "
    f"{best_threshold_row['recall']:.4f}"
)

print(
    f"Specificity OOF: "
    f"{best_threshold_row['specificity']:.4f}"
)

print(
    f"F1 OOF: "
    f"{best_threshold_row['f1']:.4f}"
)

print(
    f"MCC OOF: "
    f"{best_threshold_row['mcc']:.4f}"
)

In [ ]:
plt.figure(
    figsize=(10, 6)
)

plt.plot(
    threshold_report["threshold"],
    threshold_report["precision"],
    label="Precision"
)

plt.plot(
    threshold_report["threshold"],
    threshold_report["recall"],
    label="Recall"
)

plt.plot(
    threshold_report["threshold"],
    threshold_report["specificity"],
    label="Specificity"
)

plt.plot(
    threshold_report["threshold"],
    threshold_report["f1"],
    label="F1"
)

plt.axvline(
    best_threshold,
    linestyle="--",
    label=(
        f"Threshold seleccionado "
        f"({best_threshold:.3f})"
    )
)

plt.xlabel("Threshold")
plt.ylabel("Métrica")

plt.title(
    "Precision, Recall y Specificity "
    "según threshold"
)

plt.legend()

plt.tight_layout()

plt.show()

In [ ]:
display(
    threshold_strategies.sort_values(
        "precision",
        ascending=False
    )
)

## 9. Ajuste final y evaluación holdout V3


In [ ]:
# Threshold definitivo seleccionado mediante análisis OOF
best_threshold = 0.3993

print(
    f"Threshold definitivo: "
    f"{best_threshold:.4f}"
)

In [ ]:

if best_model_family == "XGBoost":

    final_model = clone(
        xgb_search.best_estimator_
    )

    final_model.fit(
        X_dev_final,
        y_dev
    )

    holdout_probabilities = (
        final_model.predict_proba(
            X_holdout_final
        )[:, 1]
    )

else:

    final_model = CatBoostClassifier(
        **best_catboost_params
    )

    final_model.fit(
        X_dev_final,
        y_dev,
        cat_features=categorical_columns
    )

    holdout_probabilities = (
        final_model.predict_proba(
            X_holdout_final
        )[:, 1]
    )


holdout_predictions = (
    holdout_probabilities
    >= best_threshold
).astype(int)


print("Modelo final entrenado.")
print(
    f"Threshold aplicado: "
    f"{best_threshold:.4f}"
)

In [ ]:
holdout_pr_auc = average_precision_score(
    y_holdout,
    holdout_probabilities
)

holdout_roc_auc = roc_auc_score(
    y_holdout,
    holdout_probabilities
)

holdout_precision = precision_score(
    y_holdout,
    holdout_predictions,
    zero_division=0
)

holdout_recall = recall_score(
    y_holdout,
    holdout_predictions,
    zero_division=0
)

holdout_f1 = f1_score(
    y_holdout,
    holdout_predictions,
    zero_division=0
)

holdout_f2 = fbeta_score(
    y_holdout,
    holdout_predictions,
    beta=2,
    zero_division=0
)

holdout_balanced_accuracy = (
    balanced_accuracy_score(
        y_holdout,
        holdout_predictions
    )
)

holdout_mcc = matthews_corrcoef(
    y_holdout,
    holdout_predictions
)

holdout_accuracy = accuracy_score(
    y_holdout,
    holdout_predictions
)


tn, fp, fn, tp = confusion_matrix(
    y_holdout,
    holdout_predictions
).ravel()


specificity = (
    tn / (tn + fp)
    if (tn + fp) > 0
    else np.nan
)


final_metrics = pd.DataFrame({
    "metrica": [
        "PR-AUC",
        "ROC-AUC",
        "Recall",
        "Specificity",
        "Precision",
        "F1",
        "F2",
        "Balanced Accuracy",
        "MCC",
        "Accuracy"
    ],
    "valor": [
        holdout_pr_auc,
        holdout_roc_auc,
        holdout_recall,
        specificity,
        holdout_precision,
        holdout_f1,
        holdout_f2,
        holdout_balanced_accuracy,
        holdout_mcc,
        holdout_accuracy
    ]
})


display(final_metrics)

In [ ]:
print("MATRIZ DE CONFUSIÓN")
print("=" * 50)

print(
    f"TN: {tn:,} | "
    f"FP: {fp:,}"
)

print(
    f"FN: {fn:,} | "
    f"TP: {tp:,}"
)

In [ ]:
confusion = np.array([
    [tn, fp],
    [fn, tp]
])


fig, ax = plt.subplots(
    figsize=(6, 5)
)

ax.imshow(
    confusion
)

ax.set_xticks(
    [0, 1]
)

ax.set_yticks(
    [0, 1]
)

ax.set_xticklabels(
    [
        "Pred. 0",
        "Pred. 1"
    ]
)

ax.set_yticklabels(
    [
        "Real 0",
        "Real 1"
    ]
)


for i in range(2):

    for j in range(2):

        ax.text(
            j,
            i,
            str(confusion[i, j]),
            ha="center",
            va="center",
            fontsize=14
        )


ax.set_title(
    "Matriz de confusión — Holdout V3"
)

plt.tight_layout()

plt.savefig(
    REPORTS_DIR
    / "holdout_confusion_matrix_v3.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
holdout_precision_curve, holdout_recall_curve, _ = (
    precision_recall_curve(
        y_holdout,
        holdout_probabilities
    )
)


plt.figure(
    figsize=(7, 5)
)

plt.plot(
    holdout_recall_curve,
    holdout_precision_curve
)

plt.axhline(
    y_holdout.mean(),
    linestyle="--",
    label=(
        f"Baseline = "
        f"{y_holdout.mean():.3f}"
    )
)

plt.xlabel(
    "Recall"
)

plt.ylabel(
    "Precision"
)

plt.title(
    f"Precision-Recall Holdout V3 "
    f"(AP = {holdout_pr_auc:.3f})"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    REPORTS_DIR
    / "holdout_precision_recall_curve_v3.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
fpr, tpr, _ = roc_curve(
    y_holdout,
    holdout_probabilities
)


plt.figure(
    figsize=(7, 5)
)

plt.plot(
    fpr,
    tpr,
    label=(
        f"ROC-AUC = "
        f"{holdout_roc_auc:.3f}"
    )
)

plt.plot(
    [0, 1],
    [0, 1],
    linestyle="--"
)

plt.xlabel(
    "False Positive Rate"
)

plt.ylabel(
    "True Positive Rate"
)

plt.title(
    "ROC — Holdout V3"
)

plt.legend()

plt.tight_layout()

plt.savefig(
    REPORTS_DIR
    / "holdout_roc_curve_v3.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
classification_results = classification_report(
    y_holdout,
    holdout_predictions,
    output_dict=True,
    zero_division=0
)


classification_report_df = pd.DataFrame(
    classification_results
).T


display(
    classification_report_df
)

## 10. Guardado


In [ ]:
def to_python(value):

    if isinstance(
        value,
        np.generic
    ):
        return value.item()

    if isinstance(
        value,
        dict
    ):
        return {
            key: to_python(item)
            for key, item in value.items()
        }

    if isinstance(
        value,
        (list, tuple)
    ):
        return [
            to_python(item)
            for item in value
        ]

    return value

In [ ]:
if best_model_family == "XGBoost":

    FINAL_MODEL_PATH = (
        MODELS_DIR
        / "final_liver_risk_model_v3.joblib"
    )

    joblib.dump(
        final_model,
        FINAL_MODEL_PATH
    )

    best_parameters_to_save = (
        xgb_search.best_params_
    )


else:

    FINAL_MODEL_PATH = (
        MODELS_DIR
        / "final_liver_risk_model_v3.cbm"
    )

    final_model.save_model(
        str(
            FINAL_MODEL_PATH
        )
    )

    joblib.dump(
        final_model,
        MODELS_DIR
        / "final_liver_risk_model_v3.joblib"
    )

    best_parameters_to_save = (
        best_catboost_params
    )


print(
    f"Modelo guardado en: "
    f"{FINAL_MODEL_PATH}"
)

In [ ]:
benchmark_results.to_csv(
    REPORTS_DIR
    / "benchmark_models_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

benchmark_folds.to_csv(
    REPORTS_DIR
    / "benchmark_folds_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

catboost_search_results.to_csv(
    REPORTS_DIR
    / "catboost_tuning_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

optimized_results.to_csv(
    REPORTS_DIR
    / "optimized_models_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

threshold_report.to_csv(
    REPORTS_DIR
    / "threshold_analysis_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

threshold_strategies.to_csv(
    REPORTS_DIR
    / "threshold_strategies_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

final_metrics.to_csv(
    REPORTS_DIR
    / "final_holdout_metrics_v3.csv",
    index=False,
    encoding="utf-8-sig"
)

classification_report_df.to_csv(
    REPORTS_DIR
    / "classification_report_v3.csv",
    encoding="utf-8-sig"
)

In [ ]:
holdout_predictions_df = pd.DataFrame({
    ID_COLUMN: (
        df.loc[
            holdout_indices,
            ID_COLUMN
        ].to_numpy()
    ),

    "y_true":
        y_holdout.to_numpy(),

    "probability":
        holdout_probabilities,

    "prediction":
        holdout_predictions
})


holdout_predictions_df.to_csv(
    REPORTS_DIR
    / "holdout_predictions_v3.csv",
    index=False,
    encoding="utf-8-sig"
)


display(
    holdout_predictions_df.head()
)

In [ ]:
final_configuration = {

    "notebook_version":
        NOTEBOOK_VERSION,

    "model_family":
        best_model_family,

    "dataset":
        DATASET_PATH.name,

    "number_original_features":
        len(feature_columns),

    "continuous_features":
        continuous_columns,

    "categorical_features":
        categorical_columns,

    "threshold":
        float(best_threshold),

    "threshold_selection":
        "Max F1 sobre predicciones OOF de development",

    "development_samples":
        int(len(y_dev)),

    "holdout_samples":
        int(len(y_holdout)),

    "positive_development":
        int(y_dev.sum()),

    "positive_holdout":
        int(y_holdout.sum()),

    "imbalance_ratio_development":
        float(imbalance_ratio),

    "repeated_cv_splits":
        N_SPLITS,

    "repeated_cv_repeats":
        N_REPEATS,

    "oof_pr_auc":
        float(oof_pr_auc),

    "oof_roc_auc":
        float(oof_roc_auc),

    "best_parameters":
        to_python(
            best_parameters_to_save
        ),

    "holdout_metrics": {

        "pr_auc":
            float(holdout_pr_auc),

        "roc_auc":
            float(holdout_roc_auc),

        "recall":
            float(holdout_recall),

        "specificity":
            float(specificity),

        "precision":
            float(holdout_precision),

        "f1":
            float(holdout_f1),

        "f2":
            float(holdout_f2),

        "balanced_accuracy":
            float(
                holdout_balanced_accuracy
            ),

        "mcc":
            float(holdout_mcc),

        "accuracy":
            float(holdout_accuracy)
    }
}


with open(
    REPORTS_DIR
    / "final_model_configuration_v3.json",
    "w",
    encoding="utf-8"
) as file:

    json.dump(
        final_configuration,
        file,
        indent=4,
        ensure_ascii=False
    )

In [ ]:
print("ENTRENAMIENTO V3 FINALIZADO")
print("=" * 70)

print(
    f"Modelo ganador: "
    f"{best_model_family}"
)

print(
    f"Variables originales: "
    f"{len(feature_columns)}"
)

print(
    f"Threshold definitivo: "
    f"{best_threshold:.4f}"
)

print(
    "PR-AUC repeated CV: "
    f"{optimized_results.iloc[0]['pr_auc_mean']:.4f} "
    f"± "
    f"{optimized_results.iloc[0]['pr_auc_std']:.4f}"
)

print(
    f"PR-AUC OOF: "
    f"{oof_pr_auc:.4f}"
)

print(
    f"ROC-AUC OOF: "
    f"{oof_roc_auc:.4f}"
)


print("\nRESULTADOS HOLDOUT V3")
print("-" * 70)

print(
    f"PR-AUC: "
    f"{holdout_pr_auc:.4f}"
)

print(
    f"ROC-AUC: "
    f"{holdout_roc_auc:.4f}"
)

print(
    f"Precision: "
    f"{holdout_precision:.4f}"
)

print(
    f"Recall: "
    f"{holdout_recall:.4f}"
)

print(
    f"Specificity: "
    f"{specificity:.4f}"
)

print(
    f"F1: "
    f"{holdout_f1:.4f}"
)

print(
    f"F2: "
    f"{holdout_f2:.4f}"
)

print(
    f"Balanced Accuracy: "
    f"{holdout_balanced_accuracy:.4f}"
)

print(
    f"MCC: "
    f"{holdout_mcc:.4f}"
)

print(
    "\nMatriz de confusión:"
)

print(
    f"TN={tn} | FP={fp} | "
    f"FN={fn} | TP={tp}"
)

print(
    f"\nModelo guardado en:"
    f"\n{FINAL_MODEL_PATH}"
)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# IMPORTANCIA DE VARIABLES DEL MEJOR MODELO
# ============================================================

if best_model_family == "XGBoost":
    final_model = xgb_search.best_estimator_

elif best_model_family == "CatBoost":
    final_model = catboost_final_model  # Ajustaremos este nombre si el tuyo es distinto

else:
    raise ValueError(f"Modelo no reconocido: {best_model_family}")


# ------------------------------------------------------------
# Obtener nombres de variables
# ------------------------------------------------------------

if hasattr(X_dev_final, "columns"):
    feature_names = X_dev_final.columns.tolist()
else:
    feature_names = [f"feature_{i}" for i in range(X_dev_final.shape[1])]


# ------------------------------------------------------------
# Obtener importancias
# ------------------------------------------------------------

feature_importance = pd.DataFrame({
    "variable": feature_names,
    "importance": final_model.feature_importances_
})

feature_importance = (
    feature_importance
    .sort_values("importance", ascending=False)
    .reset_index(drop=True)
)

feature_importance["importance_normalized"] = (
    feature_importance["importance"]
    / feature_importance["importance"].sum()
)


display(feature_importance.head(30))


# ------------------------------------------------------------
# Gráfico Top 20
# ------------------------------------------------------------

top_features = feature_importance.head(20).sort_values(
    "importance",
    ascending=True
)

plt.figure(figsize=(10, 8))

plt.barh(
    top_features["variable"],
    top_features["importance"]
)

plt.xlabel("Importancia")
plt.ylabel("Variable")
plt.title(
    f"Top 20 variables más importantes - {best_model_family}"
)

plt.tight_layout()
plt.show()